In [1]:
import os

base_path = '//Users/laasyab/Documents/501project /input/prose'  

print(os.path.exists(base_path))  
print(os.listdir(base_path))      

True
['JaneAusten', '.DS_Store', 'FScottFitzgerald', 'WilliamShakespeare', 'AgathaChristie', 'EdgarAllenPoe', 'MarkTwain', 'ArthurConanDoyle', 'ErnestHemingway', 'RudyardKipling', 'CharlesDickens']


In [2]:
import os
import pandas as pd
import re

base_path = '//Users/laasyab/Documents/501project /input/prose'


data = []

for author in os.listdir(base_path):
    author_path = os.path.join(base_path, author)
    if os.path.isdir(author_path):
        # Loop through each .txt book file
        for book_file in os.listdir(author_path):
            if book_file.endswith('.txt'):
                file_path = os.path.join(author_path, book_file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    text = f.read()
      
                    text = re.sub(r'\s+', ' ', text)  
                    data.append({
                        'author': author,
                        'book': book_file.replace('.txt',''),
                        'text': text
                    })

df = pd.DataFrame(data)

# Quick check
print(df.head())
print("Total books loaded:", len(df))

               author                 book  \
0          JaneAusten  SenseAndSensibility   
1          JaneAusten    PrideAndPrejudice   
2    FScottFitzgerald          GreatGatsby   
3    FScottFitzgerald   BeautifulAndDamned   
4  WilliamShakespeare       RomeoAndJuliet   

                                                text  
0  ﻿The Project Gutenberg eBook of Sense and Sens...  
1  ﻿The Project Gutenberg eBook of Pride and Prej...  
2  ﻿The Project Gutenberg eBook of The Great Gats...  
3  ﻿The Project Gutenberg eBook of The Beautiful ...  
4  ﻿The Project Gutenberg eBook of Romeo and Juli...  
Total books loaded: 20


In [3]:
df['num_sentences_est'] = df['text'].apply(lambda x: len(x.split('.')))  # simple split by periods
print(df[['author', 'num_sentences_est']].head())
print("Total estimated sentences across all books:", df['num_sentences_est'].sum())

               author  num_sentences_est
0          JaneAusten               5190
1          JaneAusten               6679
2    FScottFitzgerald               3331
3    FScottFitzgerald               7956
4  WilliamShakespeare               2825
Total estimated sentences across all books: 99991


In [4]:
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt')
import random
import pandas as pd

sentence_list = []
author_list = []

[nltk_data] Downloading package punkt to /Users/laasyab/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
sentence_list = []
author_list = []

for idx, row in df.iterrows():
    sentences = sent_tokenize(row['text']) 
    sentences = [s for s in sentences if len(s) > 5]  
    sentence_list.extend(sentences)
    author_list.extend([row['author']] * len(sentences))

print("Total sentences after tokenization:", len(sentence_list))
print(sentence_list[:5])

Total sentences after tokenization: 83387
['\ufeffThe Project Gutenberg eBook of Sense and Sensibility This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever.', 'You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.org.', 'If you are not located in the United States, you will have to check the laws of the country where you are located before using this eBook.', 'Title: Sense and Sensibility Author: Jane Austen Release date: September 1, 1994 [eBook #161] Most recently updated: March 16, 2021 Language: English Other information and formats: www.gutenberg.org/ebooks/161 *** START OF THE PROJECT GUTENBERG EBOOK SENSE AND SENSIBILITY *** [Illustration] Sense and Sensibility by Jane Austen (1811) Contents CHAPTER I CHAPTER II CHAPTER III CHAPTER IV CHAPTER V CHAPTER VI CHAPTER VII CHAPTER VIII CHAPTER IX

In [6]:
combined = list(zip(sentence_list, author_list))
random.seed(42)
random.shuffle(combined)
sentence_list, author_list = zip(*combined)

print("First 5 shuffled sentences with authors:")
for i in range(5):
    print(author_list[i], ":", sentence_list[i])

First 5 shuffled sentences with authors:
ErnestHemingway : “We’re going home!” “It would be fine if we all went home,” Piani said.
ErnestHemingway : It smelled of new furniture.
JaneAusten : He seems an excellent man; and I think him uncommonly pleasing.” “So do I.
FScottFitzgerald : "Well, I haven't any desire to go to the Barneses."
WilliamShakespeare : Now cracks a noble heart.


In [7]:
sentence_df = pd.DataFrame({
    'text': sentence_list,
    'author': author_list
})

print(sentence_df.head())
print("Total sentences in DataFrame:", len(sentence_df))

                                                text              author
0  “We’re going home!” “It would be fine if we al...     ErnestHemingway
1                       It smelled of new furniture.     ErnestHemingway
2  He seems an excellent man; and I think him unc...          JaneAusten
3  "Well, I haven't any desire to go to the Barne...    FScottFitzgerald
4                          Now cracks a noble heart.  WilliamShakespeare
Total sentences in DataFrame: 83387


In [8]:
import string 

cleaned_sentences = []

for sent in sentence_df['text']:
    sent_clean = sent.lower()  
    sent_clean = sent_clean.translate(str.maketrans('', '', string.punctuation))  
    sent_clean = sent_clean.replace('“', '').replace('”', '')  
    sent_clean = sent_clean.replace('—', '').replace('–', '')  
    sent_clean = sent_clean.replace('…', '')  
    cleaned_sentences.append(sent_clean)

sentence_df['clean_text'] = cleaned_sentences
print(sentence_df.head())

                                                text              author  \
0  “We’re going home!” “It would be fine if we al...     ErnestHemingway   
1                       It smelled of new furniture.     ErnestHemingway   
2  He seems an excellent man; and I think him unc...          JaneAusten   
3  "Well, I haven't any desire to go to the Barne...    FScottFitzgerald   
4                          Now cracks a noble heart.  WilliamShakespeare   

                                          clean_text  
0  we’re going home it would be fine if we all we...  
1                        it smelled of new furniture  
2  he seems an excellent man and i think him unco...  
3     well i havent any desire to go to the barneses  
4                           now cracks a noble heart  


In [9]:
sentence_df['word_count'] = sentence_df['clean_text'].apply(lambda x: len(x.split()))
sentence_df['char_count'] = sentence_df['clean_text'].apply(lambda x: len(x))

print("Sentence length statistics (words):")
print(sentence_df['word_count'].describe())

print("Sentence length statistics (characters):")
print(sentence_df['char_count'].describe())

Sentence length statistics (words):
count    83387.000000
mean        19.314749
std         17.348981
min          1.000000
25%          7.000000
50%         15.000000
75%         26.000000
max        299.000000
Name: word_count, dtype: float64
Sentence length statistics (characters):
count    83387.000000
mean       100.906820
std         93.477933
min          2.000000
25%         37.000000
50%         74.000000
75%        136.000000
max       1732.000000
Name: char_count, dtype: float64


In [10]:
sentence_df.to_csv('sentence_dataset.csv', index=False)
print("Preprocessed dataset saved!")

Preprocessed dataset saved!


In [11]:
sentence_df['sentence_length'] = sentence_df['text'].apply(lambda x: len(x.split()))

mean_len = sentence_df['sentence_length'].mean()
median_len = sentence_df['sentence_length'].median()
min_len = sentence_df['sentence_length'].min()
max_len = sentence_df['sentence_length'].max()
std_len = sentence_df['sentence_length'].std()

print("Mean:", mean_len)
print("Median:", median_len)
print("Min:", min_len)
print("Max:", max_len)
print("Std Dev:", std_len)

Mean: 19.32172880664852
Median: 15.0
Min: 1
Max: 299
Std Dev: 17.356841752545797
